# HelpDesk Lite — FastAPI Multi-Agent (Python)

Multi-agent IT / HR / Facilities help-desk for the Tips Hindawi Challenge (June–July 2026).

| Path | Role |
|------|------|
| `helpdesk/` | Core package — retrieval, providers, orchestrator, API, UI |
| `streamlit_app.py` | Entrypoint re-exporting `api` + Streamlit launcher |
| `helpdesk_lite.ipynb` | This notebook — walk through agents & call the API |

**Pipeline:** parallel retrieval → triage → knowledge → resolution → evaluator → human approve/reject

```bash
pip install -r requirements.txt
uvicorn streamlit_app:api --reload --port 8001   # API
streamlit run streamlit_app.py                   # UI
python evaluate.py                               # golden-set eval
```


## 1. Install / import the pipeline

In [1]:
%pip install -q -r requirements.txt

from copy import deepcopy
from streamlit_app import (
    DEFAULT_TICKETS,
    HANDBOOK_CHUNKS,
    MockProvider,
    run_multi_agent,
    select_provider,
    status_metrics,
    api,
)

print("FastAPI app:", api.title)
print("Handbook chunks:", len(HANDBOOK_CHUNKS),
      "sources:", sorted({c["sourceType"] for c in HANDBOOK_CHUNKS}))
print("Routes:", [r.path for r in api.routes if hasattr(r, "path")])

Note: you may need to restart the kernel to use updated packages.
FastAPI app: HelpDesk Lite
Handbook chunks: 14 sources: ['pdf', 'txt']
Routes: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/tickets', '/tickets', '/tickets/{ticket_id}', '/tickets/{ticket_id}/agents/run', '/tickets/{ticket_id}/agents/run', '/tickets/{ticket_id}/agents/decision', '/metrics']


## 2. Run the multi-agent orchestrator (in-process)

In [2]:
tickets = deepcopy(DEFAULT_TICKETS)
ticket = tickets[0]  # WiFi ticket

provider = select_provider("auto")  # ollama if up, else mock
print("Provider:", provider.name)

result = run_multi_agent(ticket, tickets, provider)
print(f"Duration: {result['durationMs']} ms")
print("Triage:", result["triage"])
print("Draft:", result["resolution"].get("draftResponse", "")[:200], "…")
print("Evaluator approved:", result["evaluation"].get("approved"))
print("\n--- report ---\n")
print(result["report"])

Provider: ollama:qwen2.5:7b
Duration: 21110 ms
Triage: {'category': 'OTHER', 'priority': 'MEDIUM', 'confidence': 0.75, 'rationale': "The issue described is related to network connectivity and not directly aligned with HR or Facilities categories. It could be an IT issue, but since it's not explicitly about hardware or software problems, it falls under OTHER. The frequency of disconnections suggests a medium priority.", 'tags': ['network', 'WiFi', 'disconnection']}
Draft: To address the issue of your laptop dropping the office WiFi every few minutes, please follow these steps:
1. Verify that you are connected to the correct corporate SSID.
2. Forget and re-add the netw …
Evaluator approved: True

--- report ---

# AI brief — HD-101

**Category:** OTHER · **Priority:** MEDIUM

## Summary


## Draft reply
To address the issue of your laptop dropping the office WiFi every few minutes, please follow these steps:
1. Verify that you are connected to the correct corporate SSID.
2. Forget and r

## 3. Call FastAPI with TestClient (no separate server needed)

In [4]:
from fastapi.testclient import TestClient

client = TestClient(api)

print("GET /health", client.get("/health").json())
print("GET /metrics", client.get("/metrics").json())

created = client.post(
    "/tickets",
    json={
        "title": "VPN disconnects every 10 minutes",
        "description": "Home office VPN drops after ~10 min. Need IT workaround.",
        "category": "IT",
    },
).json()
print("Created:", created["id"], created["title"])

run = client.post(
    f"/tickets/{created['id']}/agents/run",
    json={"provider": "mock"},
).json()
print("Agent provider:", run["provider"], "·", run["durationMs"], "ms")
print("Category/priority:", run["triage"].get("category"), run["triage"].get("priority"))

decision = client.post(
    f"/tickets/{created['id']}/agents/decision",
    json={"decision": "APPROVED"},
).json()
print("After approve:", decision["ticket"]["category"], decision["ticket"]["priority"])
print("Decision:", decision["run"]["decision"])

GET /health {'ok': True, 'handbookChunks': 14, 'db': '/media/nagah/01DB54578FFCD100/2026-summer/Digitera/3/helpdesk-lite-python/helpdesk.db'}
GET /metrics {'tickets': {'OPEN': 3, 'IN_PROGRESS': 1, 'RESOLVED': 1, 'CLOSED': 0}, 'agentRuns': 4, 'pendingDecisions': 2, 'handbookChunks': 14}
Created: HD-035A85 VPN disconnects every 10 minutes
Agent provider: mock · 1 ms
Category/priority: IT MEDIUM
After approve: IT MEDIUM
Decision: APPROVED


## 4. Optional — hit a live uvicorn server

In a terminal: `uvicorn streamlit_app:api --port 8001`

Then run the cell below.

In [5]:
import httpx

BASE = "http://127.0.0.1:8001"
try:
    r = httpx.get(f"{BASE}/health", timeout=2.0)
    print("Live API:", r.json())
    print("Open docs:", f"{BASE}/docs")
except httpx.HTTPError as e:
    print("API not running (start uvicorn).", e)

Live API: {'ok': True, 'handbookChunks': 14, 'db': '/media/nagah/01DB54578FFCD100/2026-summer/Digitera/3/helpdesk-lite-python/helpdesk.db'}
Open docs: http://127.0.0.1:8001/docs


## 5. Evaluation + Streamlit

```bash
python evaluate.py                 # golden set → outputs/eval/
streamlit run streamlit_app.py     # UI
```

Roles: **Employee** (submit) → **Support** (run agents + approve) → **Manager** (metrics / briefs).

After **Run agents**, check `outputs/reports/`, `outputs/emails/`, `outputs/webhooks/` for automated actions.

Agents never silently close tickets; support must approve category/priority.